In [1]:
import pandas as pd
import sqlite3
from pathlib import Path

# Paths
BASE_DIR = Path().resolve()
PROCESSED_DIR = BASE_DIR / "data" / "processed"
DB_PATH = BASE_DIR / "data" / "db" / "bluestock_mf.db"

# Connect to SQLite database (creates it if not exists)
conn = sqlite3.connect(DB_PATH)
print("✅ Database connected!")

# Load all cleaned files
fund_master = pd.read_csv(PROCESSED_DIR / "01_fund_master.csv")
nav_history = pd.read_csv(PROCESSED_DIR / "02_nav_history.csv")
aum = pd.read_csv(PROCESSED_DIR / "03_aum_by_fund_house.csv")
sip = pd.read_csv(PROCESSED_DIR / "04_monthly_sip_inflows.csv")
performance = pd.read_csv(PROCESSED_DIR / "07_scheme_performance.csv")
transactions = pd.read_csv(PROCESSED_DIR / "08_investor_transactions.csv")
holdings = pd.read_csv(PROCESSED_DIR / "09_portfolio_holdings.csv")
benchmark = pd.read_csv(PROCESSED_DIR / "10_benchmark_indices.csv")
metrics = pd.read_csv(PROCESSED_DIR / "performance_metrics.csv")

print("✅ All files loaded!")

✅ Database connected!
✅ All files loaded!


In [2]:
# Save all tables to SQLite database
fund_master.to_sql("fund_master", conn, if_exists="replace", index=False)
nav_history.to_sql("nav_history", conn, if_exists="replace", index=False)
aum.to_sql("aum_by_fund_house", conn, if_exists="replace", index=False)
sip.to_sql("monthly_sip_inflows", conn, if_exists="replace", index=False)
performance.to_sql("scheme_performance", conn, if_exists="replace", index=False)
transactions.to_sql("investor_transactions", conn, if_exists="replace", index=False)
holdings.to_sql("portfolio_holdings", conn, if_exists="replace", index=False)
benchmark.to_sql("benchmark_indices", conn, if_exists="replace", index=False)
metrics.to_sql("performance_metrics", conn, if_exists="replace", index=False)

print("✅ All tables saved to database!")

✅ All tables saved to database!


In [3]:
# Verify all tables are in the database
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print("Tables in database:")
print(tables)

# Run a sample query
print("\nTop 5 funds by NAV:")
query = """
SELECT f.scheme_name, f.fund_house, n.nav, n.date
FROM fund_master f
JOIN nav_history n ON f.amfi_code = n.amfi_code
ORDER BY n.nav DESC
LIMIT 5
"""
result = pd.read_sql(query, conn)
print(result)

Tables in database:
                    name
0            fund_master
1            nav_history
2      aum_by_fund_house
3    monthly_sip_inflows
4     scheme_performance
5  investor_transactions
6     portfolio_holdings
7      benchmark_indices
8    performance_metrics

Top 5 funds by NAV:
                            scheme_name         fund_house        nav  \
0  Kotak Liquid Fund - Regular - Growth  Kotak Mahindra MF  4268.5497   
1  Kotak Liquid Fund - Regular - Growth  Kotak Mahindra MF  4266.9021   
2  Kotak Liquid Fund - Regular - Growth  Kotak Mahindra MF  4264.9256   
3  Kotak Liquid Fund - Regular - Growth  Kotak Mahindra MF  4262.3302   
4  Kotak Liquid Fund - Regular - Growth  Kotak Mahindra MF  4260.3220   

         date  
0  2026-05-29  
1  2026-05-28  
2  2026-05-27  
3  2026-05-26  
4  2026-05-22  


In [4]:
# Save schema to sql folder
schema = """
CREATE TABLE IF NOT EXISTS fund_master (
    amfi_code INTEGER,
    fund_house TEXT,
    scheme_name TEXT,
    category TEXT,
    sub_category TEXT,
    plan TEXT,
    launch_date TEXT,
    benchmark TEXT,
    expense_ratio_pct REAL,
    exit_load_pct REAL,
    min_sip_amount REAL,
    min_lumpsum_amount REAL,
    fund_manager TEXT,
    risk_category TEXT,
    sebi_category_code TEXT
);

CREATE TABLE IF NOT EXISTS nav_history (
    amfi_code INTEGER,
    date TEXT,
    nav REAL
);

CREATE TABLE IF NOT EXISTS performance_metrics (
    amfi_code INTEGER,
    scheme_name TEXT,
    cagr_pct REAL,
    sharpe_ratio REAL,
    beta REAL,
    var_95_pct REAL
);
"""

SQL_DIR = BASE_DIR / "sql"
with open(SQL_DIR / "schema.sql", "w") as f:
    f.write(schema)

print("✅ schema.sql saved!")

✅ schema.sql saved!
